# 3— Test Evaluation

Requirements covered:
- Precision
- Recall
- F1-score
- Confusion matrix


In [ ]:
from pathlib import Path
import json, time
import torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.models import efficientnet_b0
from sklearn.metrics import classification_report,confusion_matrix,ConfusionMatrixDisplay,precision_score,recall_score,f1_score,accuracy_score

ROOT=Path.cwd(); DATA=ROOT/"data"; MODELS=ROOT/"models"; RESULTS=ROOT/"results"
RESULTS.mkdir(exist_ok=True)
DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")

tf=transforms.Compose([
    transforms.Resize((224,224)),transforms.ToTensor(),
    transforms.Normalize([.485,.456,.406],[.229,.224,.225])
])

test_ds=datasets.ImageFolder(DATA/"test",transform=tf)
test_loader=DataLoader(test_ds,batch_size=16,shuffle=False,num_workers=0)

model=efficientnet_b0(weights=None)
model.classifier[1]=torch.nn.Linear(model.classifier[1].in_features,len(test_ds.classes))
model.load_state_dict(torch.load(MODELS/"best_model.pth",map_location=DEVICE))
model.to(DEVICE).eval()

print("Classes:",test_ds.classes)
print("Test images:",len(test_ds))


In [ ]:
y_true=[]; y_pred=[]; confidences=[]
with torch.no_grad():
    for x,y in test_loader:
        prob=torch.softmax(model(x.to(DEVICE)),1)
        y_true+=y.tolist()
        y_pred+=prob.argmax(1).cpu().tolist()
        confidences+=prob.max(1).values.cpu().tolist()


In [ ]:
print(classification_report(y_true,y_pred,target_names=test_ds.classes,zero_division=0))

metrics={
    "accuracy":accuracy_score(y_true,y_pred),
    "weighted_precision":precision_score(y_true,y_pred,average="weighted",zero_division=0),
    "weighted_recall":recall_score(y_true,y_pred,average="weighted",zero_division=0),
    "weighted_f1":f1_score(y_true,y_pred,average="weighted",zero_division=0),
}
print(metrics)


In [ ]:
cm=confusion_matrix(y_true,y_pred)
print("Confusion matrix:\n",cm)

disp=ConfusionMatrixDisplay(confusion_matrix=cm,display_labels=test_ds.classes)
disp.plot()
plt.title("Bottle Defect Classifier — Confusion Matrix")
plt.tight_layout()
plt.savefig(RESULTS/"confusion_matrix.png",dpi=200)
plt.show()

RESULTS.joinpath("evaluation_metrics.json").write_text(
    json.dumps({**metrics,"confusion_matrix":cm.tolist()},indent=2)
)
